In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "vscode"

# --- Calcul des moyennes par modèle ---

# V0 - Random Forest
v0 = pd.read_csv("results_v0.csv")
v0_mean = v0[["WMAE","MAE","RMSE","R2"]].mean()

# V2 - Random Forest
v2 = pd.read_csv("results_v2.csv")
v2_mean = v2[["WMAE","MAE","RMSE","R2"]].mean()

# XGBoost V2
xgb = pd.read_csv("XGB_results.csv")
xgb_clean = xgb[xgb["Fold"] != "Moyenne"]
xgb_mean = xgb_clean[["WMAE","MAE","RMSE","R2"]].astype(float).mean()

# LightGBM Optuna
lgbm = pd.read_csv("results_lgbm_optuna.csv")
lgbm_clean = lgbm[lgbm["Fold"] != "Moyenne"]
lgbm_mean = lgbm_clean[["WMAE","MAE","RMSE","R2"]].astype(float).mean()

# XGBoost + sin/cos
xgb_sc = pd.read_csv("results_xgb_sincos.csv")
xgb_sc_mean = xgb_sc[["WMAE","MAE","RMSE","R2"]].iloc[0]

# --- Tableau comparatif ---
df_graph = pd.DataFrame([
    {"Variante": "V0 - Random Forest",  "WMAE": v0_mean["WMAE"],     "MAE": v0_mean["MAE"],     "RMSE": v0_mean["RMSE"],     "R2": v0_mean["R2"]},
    {"Variante": "V2 - Random Forest",  "WMAE": v2_mean["WMAE"],     "MAE": v2_mean["MAE"],     "RMSE": v2_mean["RMSE"],     "R2": v2_mean["R2"]},
    {"Variante": "V2 - XGB - XGBoost",    "WMAE": xgb_mean["WMAE"],    "MAE": xgb_mean["MAE"],    "RMSE": xgb_mean["RMSE"],    "R2": xgb_mean["R2"]},
    {"Variante": "V2 -LightGBM Optuna",     "WMAE": lgbm_mean["WMAE"],   "MAE": lgbm_mean["MAE"],   "RMSE": lgbm_mean["RMSE"],   "R2": lgbm_mean["R2"]},
    {"Variante": "VF -XGBoost + sin/cos*",  "WMAE": xgb_sc_mean["WMAE"], "MAE": xgb_sc_mean["MAE"], "RMSE": xgb_sc_mean["RMSE"], "R2": xgb_sc_mean["R2"]},
])

# --- Graphique ---
metrics   = ["WMAE", "MAE", "RMSE", "R2"]
positions = [(1,1), (1,2), (2,1), (2,2)]
bar_colors = ["#636EFA","#636EFA","#636EFA","#636EFA","#FFA15A"]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("WMAE", "MAE", "RMSE", "R²"),
    vertical_spacing=0.14,
    horizontal_spacing=0.08
)

for i, (metric, (r, c)) in enumerate(zip(metrics, positions), start=1):
    fig.add_trace(
        go.Bar(
            x            = df_graph["Variante"],
            y            = df_graph[metric].round(2),
            text         = df_graph[metric].round(2),
            textposition = "outside",
            textfont     = dict(size=9),
            marker_color = bar_colors
        ),
        row=r, col=c
    )

    ymax = df_graph[metric].max()
    fig.update_yaxes(range=[0, ymax * 1.25], row=r, col=c)

    # Flèche V0 → XGBoost + sin/cos
    v0_val   = df_graph.iloc[0][metric]
    best_val = df_graph.iloc[4][metric]

    variation = (best_val - v0_val) / abs(v0_val) * 100
    is_good   = variation > 0 if metric == "R2" else variation < 0
    color_arr = "green" if is_good else "red"

    xref = "x" if i == 1 else f"x{i}"
    yref = "y" if i == 1 else f"y{i}"

    fig.add_annotation(
        x=df_graph.iloc[4]["Variante"], y=best_val,
        ax=df_graph.iloc[0]["Variante"], ay=v0_val,
        xref=xref, yref=yref, axref=xref, ayref=yref,
        text="", showarrow=True,
        arrowhead=3, arrowsize=1, arrowwidth=2, arrowcolor=color_arr
    )
    fig.add_annotation(
        x=df_graph.iloc[4]["Variante"],
        y=(v0_val + best_val) / 2 + ymax * 0.05,
        xref=xref, yref=yref,
        text=f"{variation:+.1f}%",
        showarrow=False,
        font=dict(size=11, color=color_arr),
        bgcolor="white", bordercolor=color_arr, borderwidth=1
    )

fig.update_layout(
    height=700, width=1000,
    showlegend=False,
    template="plotly_white",
    title="Comparaison des modèles - Walmart Sales Forecast",
    title_font_size=16,
    margin=dict(t=100, b=80)
)



fig.update_xaxes(tickangle=-20)
fig.show()


## Comparaison des modèles — Analyse des résultats
 
### Meilleur modèle - XGBoost + sin/cos
- WMAE : 1 591$ — meilleur résultat de tous les modèles testés
- R² : 0.97 — variance expliquée la plus élevée
- MAE : 1 521$ et RMSE : 3 505$ — erreurs les plus faibles
- L'encodage circulaire de la semaine (Week_sin, Week_cos) capture
  la saisonnalité cyclique mieux qu'un entier brut
 
### Progression globale
- V0 baseline : 4 249$ → XGBoost + sin/cos : 1 591$ soit -62.6% sur le WMAE
- R² passe de 0.82 à 0.97 soit +18.9% — le modèle explique
  97% de la variance des ventes hebdomadaires
 
### Apport de chaque étape
- V2 Random Forest : feature engineering + transformation log → -46.1%
- XGBoost V2 : changement d'algorithme → légère amélioration
- LightGBM Optuna : sélection SHAP + optimisation Optuna → 2 095$
- XGBoost + sin/cos : encodage temporel circulaire → 1 591$
 
 